In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
import datetime as dt
import Engine as Ctool
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import math
from scipy.optimize import minimize
import cvxpy as cp
from sklearn.linear_model import LinearRegression

In [2]:
date = dt.date(2026, 4, 1)
index = "SP500"
lbklen = "-50b"
end_date = date
start_date = Ctool.calc_add_date(end_date, lbklen)

In [5]:
universe = Ctool.load_universe(index, date)[index]
master_index = Ctool.calc_business_date(Ctool.calc_add_date(end_date, "-5b"), end_date)
alpha_dict = {}
for ticker in universe:
    ticker_data = Ctool.calc_forward_close(ticker, start_date, end_date)[ticker]
    ticker_data['value'] = ticker_data['adjusted_close'] - ticker_data['adjusted_close'].rolling(30).mean()
    ticker_data['mean'] = ticker_data['value'].rolling(5).mean()
    ticker_data['std'] = ticker_data['value'].rolling(5).std()
    ticker_data['alpha'] = (ticker_data['value'] - ticker_data['mean']) / ticker_data['std']
    alpha_dict[ticker] = ticker_data['alpha']

alphas = pd.DataFrame(alpha_dict, index=master_index).dropna(axis=1, how='any')
columns = alphas.columns
alphas = alphas.apply(Ctool.calc_zscore, axis=1, result_type='expand')
alphas.columns = columns

In [27]:
def backtest_return(alphas, lbklen_std='20b'):
    date = alphas.index[-1]
    tickers = list(alphas.columns)
    start_date = Ctool.calc_add_date(date, "-"+lbklen_std)
    end_date = Ctool.calc_add_date(date, "-1b")
    returns = Ctool.calc_dict2df(Ctool.load_return(tickers, start_date, date, "1b")).dropna(axis=1, how='any')
    tickers = list(set(tickers)&set(returns.columns.tolist()))
    returns = returns.loc[:, tickers]
    alphas = alphas.loc[:, tickers]
    start_date_regression = max(pd.Timestamp(alphas.index[0]).date(), pd.Timestamp(returns.index[0]).date())
    X = alphas.loc[start_date_regression:end_date, :].values.flatten().reshape(-1, 1)
    y = returns.loc[start_date_regression:end_date, :].values.flatten()
    model = LinearRegression(fit_intercept=True)
    model.fit(X, y)
    ret_hat = model.predict(alphas.loc[date, :].values.flatten().reshape(-1, 1))
    cov = returns.loc[start_date:end_date, :].cov().values
    w = cp.Variable(len(tickers))
    var = cp.quad_form(w, cov)
    ret = ret_hat @ w
    objective_cvxpy = cp.Minimize(0.5 * var - ret)
    constraints_cvxpy = [cp.sum(w) == 1, w >= 0]
    prob = cp.Problem(objective_cvxpy, constraints_cvxpy)
    prob.solve(solver=cp.OSQP)
    w = np.clip(w.value, 0, 1)
    w = w / np.sum(w)
    position = dict(zip(tickers, w.tolist()))
    weights = np.array([position[t] for t in returns.columns])
    ret = (returns.loc[date, :].values @ weights).item()
    return position, ret

position, ret = backtest_return(alphas)